# Virtual Try-On (Colab/Kaggle, GitHub Clone Flow)

Use this notebook when you clone your repo in Colab/Kaggle and want image/video testing.


In [ ]:
import os, sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle')
print('Colab:', IS_COLAB, '| Kaggle:', IS_KAGGLE)


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Mothieram/tryon.git'   # change if needed
REPO_DIR_NAME = 'Tryon'
REPO_BRANCH = ''   # leave empty to auto-detect default branch

IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle')
base = Path('/content') if IS_COLAB else (Path('/kaggle/working') if IS_KAGGLE else Path.cwd())
PROJECT_DIR = base / REPO_DIR_NAME

def run(cmd, cwd=None):
    print('>>', ' '.join(cmd))
    p = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout.strip())
    if p.returncode != 0:
        if p.stderr:
            print(p.stderr.strip())
        raise RuntimeError(f'Command failed ({p.returncode}): {' '.join(cmd)}')
    return p

if not REPO_BRANCH:
    probe = subprocess.run(['git','ls-remote','--symref',REPO_URL,'HEAD'], text=True, capture_output=True)
    if probe.returncode != 0:
        print(probe.stderr)
        raise RuntimeError('Could not read remote HEAD. Check REPO_URL and repo visibility.')
    branch_line = next((ln for ln in probe.stdout.splitlines() if ln.startswith('ref: ')), '')
    REPO_BRANCH = branch_line.split('refs/heads/')[-1].split('	')[0] if 'refs/heads/' in branch_line else 'main'

print('Using branch:', REPO_BRANCH)

if PROJECT_DIR.exists():
    run(['git','fetch','--all','--prune'], cwd=PROJECT_DIR)
    run(['git','checkout',REPO_BRANCH], cwd=PROJECT_DIR)
    run(['git','pull'], cwd=PROJECT_DIR)
else:
    run(['git','clone','--branch',REPO_BRANCH,REPO_URL,str(PROJECT_DIR)])

assert (PROJECT_DIR / 'engine').exists(), f'engine/ not found in {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Using PROJECT_DIR =', PROJECT_DIR)


In [ ]:
%%bash
set -e
python -m pip install --upgrade pip
python -m pip install opencv-contrib-python ultralytics scipy scikit-image scikit-learn colorlog matplotlib imageio tqdm pyyaml psutil requests


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from engine.render_pipeline import RenderPipeline

pipeline = RenderPipeline(
    pose_model='yolov8n-pose.pt',
    device='auto',
    target_fps=30,
    enable_shadows=True,
    enable_lighting=True,
    opacity=0.95,
)

ok = pipeline.load_models()
print('load_models:', ok)
if not ok:
    raise RuntimeError('Model loading failed')

count = pipeline.load_garments(str(PROJECT_DIR / 'assets' / 'shirts'))
print('garments loaded:', count)
print('garments:', pipeline.get_shirt_names())


In [ ]:
def show_bgr(img_bgr, title='image', figsize=(8, 5)):
    plt.figure(figsize=figsize)
    plt.title(title)
    plt.axis('off')
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.show()

def run_on_image(image_path, shirt_index=0):
    pipeline.select_shirt(shirt_index)
    frame = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if frame is None:
        raise FileNotFoundError(image_path)
    out, stats = pipeline.process_frame(frame)
    return frame, out, stats

def run_on_video(video_path, out_path='outputs/tryon_out.mp4', shirt_index=0, max_frames=None):
    pipeline.select_shirt(shirt_index)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open video: {video_path}')

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 1 or np.isnan(fps):
        fps = 25.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    idx = 0
    last_stats = None
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        processed, stats = pipeline.process_frame(frame)
        writer.write(processed)
        last_stats = stats
        idx += 1
        if max_frames is not None and idx >= max_frames:
            break

    cap.release()
    writer.release()
    return out_path, idx, last_stats


In [ ]:
IMG_PATH = PROJECT_DIR / 'person.jpg'   # <-- change if needed
SHIRT_INDEX = 0

orig, pred, stats = run_on_image(IMG_PATH, shirt_index=SHIRT_INDEX)
print('stats:', stats)
show_bgr(orig, 'Original')
show_bgr(pred, 'Try-On Output')

Path('outputs').mkdir(exist_ok=True)
cv2.imwrite('outputs/tryon_image_output.png', pred)
print('Saved: outputs/tryon_image_output.png')


In [ ]:
VIDEO_PATH = PROJECT_DIR / 'input.mp4'  # <-- set your video path
if VIDEO_PATH.exists():
    out_path, n, last_stats = run_on_video(VIDEO_PATH, out_path='outputs/tryon_video_output.mp4', shirt_index=0)
    print(f'Processed {n} frame(s)')
    print('Last stats:', last_stats)
    print('Saved:', out_path)
else:
    print('Video not found, skipped:', VIDEO_PATH)


In [ ]:
# Optional cleanup
# pipeline.unload_models()
